# 🚂 RailGuard AI — V6 High-Precision Training
**EfficientNet-B3 + 5-Level FPN | Horizon-Aware Sky-Suppression Loss**

### What's New vs V5:
- **EfficientNet-B3 backbone** (~12M params vs ~1M) — far better on distant curves
- **Horizon-Aware Loss** — explicit sky/ceiling suppression penalty baked into training
- **Richer augmentation** — gamma, shadow stripes, saturation jitter, horizontal shifts
- **Gradient clipping + LR warmup** — stable convergence from epoch 1

### Steps:
1. Upload `railguard_colab_package.zip` to your Google Drive
2. Run all cells top to bottom
3. Download `rail_heatmap_v6_512x288.onnx` from your Drive when done

In [ ]:
# ─── Cell 1: Mount Drive & Install Dependencies ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', '-q', 'timm==0.9.16', 'onnxruntime'], check=True)
print('Dependencies installed!')

In [ ]:
# ─── Cell 2: Unpack Dataset ───────────────────────────────────────────────
import os

# !! Set this to the path of your zip in Google Drive !!
DRIVE_ZIP  = '/content/drive/MyDrive/railguard_colab_package.zip'
DATA_ROOT  = '/content/railguard/dataset'
OUTPUT_DIR = '/content/railguard/output_v6'

os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

import subprocess
subprocess.run(['unzip', '-q', '-o', DRIVE_ZIP, '-d', DATA_ROOT], check=True)
print('Dataset unpacked!')
print('Files:', os.listdir(DATA_ROOT)[:10])

In [ ]:
# ─── Cell 3: Config & Imports ────────────────────────────────────────────
import json, math, time, random
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm

INPUT_W      = 512
INPUT_H      = 288
BATCH_SIZE   = 16
NUM_EPOCHS   = 100
LR           = 5e-4
WEIGHT_DECAY = 2e-4

# Ground plane horizon: rail activations MUST be zero above this row
HORIZON_ROW = int(INPUT_H * 0.35)   # row 100 out of 288

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}  |  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# ─── Cell 4: Dataset with Rich Augmentation ──────────────────────────────
def render_heatmap_channel(polylines, w=INPUT_W, h=INPUT_H, sigma=2.5):
    mask = np.zeros((h, w), dtype=np.float32)
    for poly in polylines:
        pts = np.array([[int(p[0]*w), int(p[1]*h)] for p in poly], dtype=np.int32)
        if len(pts) >= 2:
            cv2.polylines(mask, [pts], False, 1.0, 3, cv2.LINE_AA)
    if sigma > 0:
        k = int(math.ceil(sigma * 3)) * 2 + 1
        mask = cv2.GaussianBlur(mask, (k, k), sigma)
        mx = mask.max()
        if mx > 0: mask /= mx
    return mask

class RailDatasetV6(Dataset):
    def __init__(self, data_root, split='train', augment=False):
        self.augment = augment
        self.data_root = data_root
        with open(os.path.join(data_root, f'{split}_metadata.json'), 'r') as f:
            meta = json.load(f)
        self.samples = []
        for m in meta:
            p = os.path.join(data_root, 'images', os.path.splitext(m['file_name'])[0] + '.jpg')
            if os.path.exists(p):
                self.samples.append((p, m))
        print(f'[{split}] {len(self.samples)} samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, item = self.samples[idx]
        bgr = cv2.imread(img_path)
        if bgr is None:
            bgr = np.zeros((INPUT_H, INPUT_W, 3), dtype=np.uint8)
        else:
            bgr = cv2.resize(bgr, (INPUT_W, INPUT_H))

        l_polys = [p.copy() for p in item['left_polylines']]
        r_polys = [p.copy() for p in item['right_polylines']]

        if self.augment:
            # 1. Horizontal flip
            if random.random() < 0.5:
                bgr = cv2.flip(bgr, 1)
                l_polys, r_polys = (
                    [[[1-pt[0], pt[1]] for pt in p] for p in r_polys],
                    [[[1-pt[0], pt[1]] for pt in p] for p in l_polys]
                )

            # 2. Color jitter (HSV saturation + brightness)
            if random.random() < 0.7:
                hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
                hsv[:,:,1] *= random.uniform(0.65, 1.35)
                hsv[:,:,2] *= random.uniform(0.60, 1.40)
                bgr = cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2BGR)

            # 3. Random gamma (tunnel darkness / bright outdoor)
            if random.random() < 0.5:
                g = random.uniform(0.45, 1.9)
                lut = np.array([((i/255)**( 1/g))*255 for i in range(256)], dtype=np.uint8)
                bgr = cv2.LUT(bgr, lut)

            # 4. Blur
            if random.random() < 0.2:
                bgr = cv2.GaussianBlur(bgr, (5,5), 0)

            # 5. Shadow stripe (foreground occlusion)
            if random.random() < 0.25:
                y1 = random.randint(INPUT_H//2, INPUT_H-30)
                y2 = min(INPUT_H, y1 + random.randint(10, 50))
                bgr[y1:y2] = (bgr[y1:y2] * 0.35).astype(np.uint8)

            # 6. Horizontal sway
            if random.random() < 0.3:
                sh = random.randint(-12, 12)
                M  = np.float32([[1,0,sh],[0,1,0]])
                bgr = cv2.warpAffine(bgr, M, (INPUT_W, INPUT_H))
                shn = sh / INPUT_W
                l_polys = [[[max(0,min(1,pt[0]+shn)), pt[1]] for pt in p] for p in l_polys]
                r_polys = [[[max(0,min(1,pt[0]+shn)), pt[1]] for pt in p] for p in r_polys]

        hm_l = render_heatmap_channel(l_polys)
        hm_r = render_heatmap_channel(r_polys)

        # CRITICAL: zero-out any GT activations above the horizon
        hm_l[:HORIZON_ROW, :] = 0.0
        hm_r[:HORIZON_ROW, :] = 0.0

        heatmap = np.stack([hm_l, hm_r], axis=0)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        rgb = (rgb - MEAN) / STD
        return torch.from_numpy(rgb.transpose(2,0,1)), torch.from_numpy(heatmap)

In [ ]:
# ─── Cell 5: EfficientNet-B3 + 5-Level FPN Model ─────────────────────────
class ConvBNReLU(nn.Module):
    def __init__(self, ic, oc, k=3, p=1):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(ic, oc, k, padding=p, bias=False),
            nn.BatchNorm2d(oc), nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.b(x)

class RailHeatmapNetV6(nn.Module):
    """EfficientNet-B3 + 5-level FPN. ~12M params — significantly more precise on distant and curved rails."""
    def __init__(self):
        super().__init__()
        # EfficientNet-B3 skip channel sizes: [24, 32, 48, 136, 1536]
        self.backbone = timm.create_model('efficientnet_b3', pretrained=True, features_only=True)
        ch = 96

        self.lat5 = nn.Conv2d(1536, ch, 1)
        self.lat4 = nn.Conv2d(136,  ch, 1)
        self.lat3 = nn.Conv2d(48,   ch, 1)
        self.lat2 = nn.Conv2d(32,   ch, 1)
        self.lat1 = nn.Conv2d(24,   ch, 1)

        self.s4 = nn.Sequential(ConvBNReLU(ch,ch), ConvBNReLU(ch,ch))
        self.s3 = nn.Sequential(ConvBNReLU(ch,ch), ConvBNReLU(ch,ch))
        self.s2 = nn.Sequential(ConvBNReLU(ch,ch), ConvBNReLU(ch,ch))
        self.s1 = nn.Sequential(ConvBNReLU(ch,ch), ConvBNReLU(ch,ch))

        self.head = nn.Sequential(
            ConvBNReLU(ch, 64),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBNReLU(64, 32),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBNReLU(32, 16),
            nn.Conv2d(16, 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        c1,c2,c3,c4,c5 = self.backbone(x)
        p5 = self.lat5(c5)
        p4 = self.s4(self.lat4(c4) + F.interpolate(p5, c4.shape[2:], mode='bilinear', align_corners=False))
        p3 = self.s3(self.lat3(c3) + F.interpolate(p4, c3.shape[2:], mode='bilinear', align_corners=False))
        p2 = self.s2(self.lat2(c2) + F.interpolate(p3, c2.shape[2:], mode='bilinear', align_corners=False))
        p1 = self.s1(self.lat1(c1) + F.interpolate(p2, c1.shape[2:], mode='bilinear', align_corners=False))
        return self.head(p1)

model = RailHeatmapNetV6().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ─── Cell 6: Horizon-Aware Loss ──────────────────────────────────────────
class HorizonAwareLoss(nn.Module):
    """
    Soft Dice + Focal BCE + SKY SUPPRESSION PENALTY.
    sky_w=3.0 strongly penalises any predicted rail activation above the horizon.
    This is what prevents lines from 'going up into the air' at the far distance.
    """
    def __init__(self, dice_w=1.5, bce_w=1.0, sky_w=3.0):
        super().__init__()
        self.dice_w = dice_w
        self.bce_w  = bce_w
        self.sky_w  = sky_w

    def forward(self, pred, target):
        eps = 1e-6
        # Dice
        inter = (pred * target).sum(dim=(2,3))
        union = (pred**2).sum(dim=(2,3)) + (target**2).sum(dim=(2,3))
        dice_loss = 1.0 - ((2*inter+eps)/(union+eps)).mean()

        # Focal BCE
        bce = F.binary_cross_entropy(pred, target, reduction='none')
        pt  = torch.where(target >= 0.5, pred, 1-pred)
        bce_loss = (0.75 * (1-pt)**2 * bce).mean()

        # Sky penalty — squared activations above horizon row
        sky_loss = (pred[:, :, :HORIZON_ROW, :] ** 2).mean()

        return self.dice_w*dice_loss + self.bce_w*bce_loss + self.sky_w*sky_loss

criterion = HorizonAwareLoss(dice_w=1.5, bce_w=1.0, sky_w=3.0)
print('Loss function ready (Dice + Focal BCE + Sky Suppression)')

In [ ]:
# ─── Cell 7: Training Loop ───────────────────────────────────────────────
train_ds = RailDatasetV6(DATA_ROOT, 'train', augment=True)
val_ds   = RailDatasetV6(DATA_ROOT, 'val',   augment=False)
train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# 5-epoch warmup then cosine anneal
def lr_lambda(ep):
    if ep < 5: return (ep+1)/5.0
    return 0.5*(1 + math.cos(math.pi*(ep-5)/(NUM_EPOCHS-5)))
scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_val = float('inf')
for epoch in range(1, NUM_EPOCHS+1):
    t0 = time.time()
    model.train(); tl = 0.0
    for imgs, hms in train_dl:
        imgs, hms = imgs.to(device), hms.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), hms)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        tl += loss.item() * len(imgs)
    tl /= len(train_ds)

    model.eval(); vl = 0.0
    with torch.no_grad():
        for imgs, hms in val_dl:
            imgs, hms = imgs.to(device), hms.to(device)
            vl += criterion(model(imgs), hms).item() * len(imgs)
    vl /= len(val_ds)
    scheduler.step()

    tag = ''
    if vl < best_val:
        best_val = vl
        torch.save(model.state_dict(), f'{OUTPUT_DIR}/best_model.pth')
        tag = ' <best>'
    print(f'Epoch [{epoch:3d}/{NUM_EPOCHS}] ({time.time()-t0:.0f}s) train={tl:.5f} val={vl:.5f} lr={optimizer.param_groups[0]["lr"]:.2e}{tag}')

print('Training complete!')

In [ ]:
# ─── Cell 8: Export to ONNX & Save to Google Drive ───────────────────────
import shutil

model.load_state_dict(torch.load(f'{OUTPUT_DIR}/best_model.pth', map_location=device))
model.eval()

dummy    = torch.randn(1, 3, INPUT_H, INPUT_W).to(device)
onnx_out = f'{OUTPUT_DIR}/rail_heatmap_v6_512x288.onnx'

torch.onnx.export(
    model, dummy, onnx_out,
    input_names=['image'], output_names=['heatmap'],
    dynamic_axes={'image': {0: 'batch'}, 'heatmap': {0: 'batch'}},
    opset_version=17
)

drive_out = '/content/drive/MyDrive/rail_heatmap_v6_512x288.onnx'
shutil.copy2(onnx_out, drive_out)

size_mb = os.path.getsize(drive_out) / 1024 / 1024
print(f'ONNX exported: {size_mb:.1f} MB')
print(f'Saved to Google Drive: {drive_out}')
print('Download this file and place it in C:\\RailGuardAI\\production_model\\rail_model.onnx')